# Lesson 01 - Pengenalan Agen AI

Selamat datang di pelajaran pertama dalam kursus **Agen AI untuk Pemula**!

Sebuah **agen AI** adalah program yang menggunakan model bahasa besar (LLM) sebagai mesin penalarannya dan dapat mengambil *tindakan* di dunia nyata — memanggil API, melakukan kueri ke basis data, atau menjalankan kode — untuk mencapai suatu tujuan atas nama pengguna.

Dalam notebook ini Anda akan membangun agen pertama Anda: sebuah **Agen Perjalanan** yang merekomendasikan destinasi liburan. Sepanjang perjalanan Anda akan belajar bagaimana:

1. Terhubung ke Azure AI Foundry Agent Service menggunakan **Microsoft Agent Framework**.
2. Memberikan agen sebuah **alat** — fungsi Python biasa yang bisa dipanggilnya.
3. Menjalankan agen dan memeriksa responsnya.
4. Menyajikan respons agen secara streaming token demi token.


## Setup

Sebelum menjalankan notebook ini, pastikan Anda telah:

1. **Proyek Azure AI Foundry** dengan model chat yang sudah diterapkan (misalnya `gpt-4o-mini`).
2. **Masuk dengan Azure CLI** — jalankan `az login` di terminal Anda.
3. **Mengatur variabel lingkungan yang diperlukan:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint proyek Azure AI Foundry Anda.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model yang sudah diterapkan.

Sel berikut memasang paket Python yang Anda butuhkan.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Membuat Agen Pertama Anda

Sebuah agen membutuhkan dua hal:

- **Instruksi** yang memberi tahu *siapa dia* dan *bagaimana cara berperilaku* (prompt sistem).
- **Alat** — fungsi Python yang dihiasi dengan `@tool` yang dapat dipanggil agen untuk mengambil informasi atau melakukan tindakan.

Di bawah ini kami mendefinisikan sebuah alat sederhana yang mengembalikan daftar destinasi liburan populer. Agen akan menggunakan alat ini ketika pengguna meminta rekomendasi perjalanan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Untuk pengalaman yang lebih interaktif, Anda dapat **stream** respons agen. Alih-alih menunggu balasan penuh, agen mengirimkan potongan teks saat mereka dihasilkan. Ini sangat berguna di antarmuka obrolan di mana Anda ingin menampilkan keluaran secara real time.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

Dalam pelajaran ini Anda belajar bagaimana:

- **Membuat penyedia** yang terhubung ke Azure AI Foundry Agent Service melalui `AzureAIProjectAgentProvider`.
- **Mendefinisikan alat** menggunakan dekorator `@tool` agar agen dapat memanggil fungsi Python Anda.
- **Menjalankan agen** dengan pesan pengguna dan mencetak responsnya.
- **Mengalirkan respons** untuk keluaran waktu nyata.

Dalam pelajaran berikutnya kita akan menjelajahi kerangka kerja agen dengan lebih mendalam dan belajar bagaimana memberikan agen alat yang lebih kuat serta kemampuan penalaran multi-langkah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai ketepatan, harap diperhatikan bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidaktepatan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sahih. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
